In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Source tables and validation configuration
# ---------------------------------------------------------------------------

TABLES = {
    "dim_team": f"{GOLD_SCHEMA}.dim_team",
    "dim_competition": f"{GOLD_SCHEMA}.dim_competition",
    "dim_season": f"{GOLD_SCHEMA}.dim_season",
    "dim_date": f"{GOLD_SCHEMA}.dim_date",
    "fact_team_season": f"{GOLD_SCHEMA}.fact_team_season",
    "fact_match": f"{GOLD_SCHEMA}.fact_match",
    "fact_team_match": f"{GOLD_SCHEMA}.fact_team_match",
}

BUSINESS_KEYS = {
    "dim_team": ["team_key"],
    "dim_competition": ["competition_key"],
    "dim_season": ["season_key"],
    "dim_date": ["date_key"],
    "fact_team_season": [
        "team_key",
        "competition_key",
        "season_key",
    ],
    "fact_match": ["match_key"],
    "fact_team_match": ["match_key", "team_key"],
}

FOREIGN_KEYS = {
    "fact_team_season": [
        ("team_key", "dim_team", "team_key"),
        ("competition_key", "dim_competition", "competition_key"),
        ("season_key", "dim_season", "season_key"),
    ],
    "fact_match": [
        ("home_team_key", "dim_team", "team_key"),
        ("away_team_key", "dim_team", "team_key"),
        ("competition_key", "dim_competition", "competition_key"),
        ("season_key", "dim_season", "season_key"),
        ("date_key", "dim_date", "date_key"),
    ],
    "fact_team_match": [
        ("team_key", "dim_team", "team_key"),
        ("opponent_team_key", "dim_team", "team_key"),
        ("competition_key", "dim_competition", "competition_key"),
        ("season_key", "dim_season", "season_key"),
        ("date_key", "dim_date", "date_key"),
    ],
}

results = []
reconciliation_results = []


# ---------------------------------------------------------------------------
# Validation reporting helpers
# ---------------------------------------------------------------------------

def record_check(
    category,
    check_name,
    status,
    observed,
    details="",
):
    results.append({
        "category": category,
        "check_name": check_name,
        "status": status,
        "observed": str(observed),
        "details": details,
    })


def record_reconciliation(
    season_key,
    competition_key,
    team_key,
    metric,
    match_value,
    season_value,
):
    reconciliation_results.append({
        "season_key": int(season_key),
        "competition_key": str(competition_key),
        "team_key": int(team_key),
        "metric": metric,
        "match_value": float(match_value),
        "season_value": float(season_value),
        "difference": float(match_value - season_value),
    })


# ---------------------------------------------------------------------------
# Validate table availability and load datasets
# ---------------------------------------------------------------------------

dataframes = {}
row_counts = {}

for table_name, full_name in TABLES.items():

    if not spark.catalog.tableExists(full_name):
        record_check(
            "TABLE",
            f"{table_name}_exists",
            "FAIL",
            "Missing",
            full_name,
        )
        continue

    df = spark.table(full_name)
    count = df.count()

    dataframes[table_name] = df
    row_counts[table_name] = count

    record_check(
        "TABLE",
        f"{table_name}_not_empty",
        "PASS" if count > 0 else "FAIL",
        count,
        full_name,
    )

# Remaining checks require the full Gold model.

missing_tables = set(TABLES) - set(dataframes)

if missing_tables:
    raise ValueError(
        "Required Gold tables are missing: "
        f"{sorted(missing_tables)}"
    )


# ---------------------------------------------------------------------------
# Validate business keys
# ---------------------------------------------------------------------------

for table_name, key_columns in BUSINESS_KEYS.items():

    df = dataframes[table_name]

    missing_columns = [
        col for col in key_columns
        if col not in df.columns
    ]

    if missing_columns:
        record_check(
            "KEY",
            f"{table_name}_key_columns",
            "FAIL",
            missing_columns,
        )
        continue

    null_condition = None

    for column_name in key_columns:
        condition = F.col(column_name).isNull()

        null_condition = (
            condition if null_condition is None
            else null_condition | condition
        )

    null_count = df.filter(null_condition).count()

    duplicate_groups = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    record_check(
        "KEY",
        f"{table_name}_non_null_keys",
        "PASS" if null_count == 0 else "FAIL",
        null_count,
        "Rows with at least one null key",
    )

    record_check(
        "KEY",
        f"{table_name}_unique_keys",
        "PASS" if duplicate_groups == 0 else "FAIL",
        duplicate_groups,
        "Duplicate business key combinations",
    )


# ---------------------------------------------------------------------------
# Validate fact foreign keys
# ---------------------------------------------------------------------------

for fact_name, relationships in FOREIGN_KEYS.items():

    fact_df = dataframes[fact_name]

    for fact_column, dimension_name, dimension_column in relationships:

        reference_df = (
            dataframes[dimension_name]
            .select(
                F.col(dimension_column).alias(fact_column)
            )
            .distinct()
        )

        orphan_count = (
            fact_df
            .select(fact_column)
            .distinct()
            .join(
                reference_df,
                on=fact_column,
                how="left_anti",
            )
            .count()
        )

        record_check(
            "FOREIGN_KEY",
            f"{fact_name}.{fact_column}",
            "PASS" if orphan_count == 0 else "FAIL",
            orphan_count,
            f"Reference: {dimension_name}.{dimension_column}",
        )


# ---------------------------------------------------------------------------
# Validate match and team-match record counts
# ---------------------------------------------------------------------------

matches_df = dataframes["fact_match"]
team_matches_df = dataframes["fact_team_match"]
season_df = dataframes["fact_team_season"]

match_count = row_counts["fact_match"]
team_match_count = row_counts["fact_team_match"]

expected_team_matches = 2 * match_count

record_check(
    "MATCH",
    "two_team_rows_per_match_total",
    "PASS" if team_match_count == expected_team_matches else "FAIL",
    team_match_count,
    f"Expected: {expected_team_matches}",
)


# ---------------------------------------------------------------------------
# Validate individual home and away pairs
# ---------------------------------------------------------------------------

pair_counts_df = (
    team_matches_df
    .groupBy("match_key")
    .agg(
        F.count("*").alias("team_rows"),
        F.sum(
            F.when(F.col("is_home"), 1).otherwise(0)
        ).alias("home_rows"),
        F.sum(
            F.when(~F.col("is_home"), 1).otherwise(0)
        ).alias("away_rows"),
    )
)

invalid_pairs = (
    pair_counts_df
    .filter(
        (F.col("team_rows") != 2)
        | (F.col("home_rows") != 1)
        | (F.col("away_rows") != 1)
    )
    .count()
)

record_check(
    "MATCH",
    "home_away_pair_integrity",
    "PASS" if invalid_pairs == 0 else "FAIL",
    invalid_pairs,
    "Expected exactly one home and one away row per match",
)

# Total row counts alone cannot detect a missing match
# whose place is taken by duplicated records elsewhere.

missing_team_matches = (
    matches_df
    .select("match_key")
    .join(
        team_matches_df.select("match_key").distinct(),
        on="match_key",
        how="left_anti",
    )
    .count()
)

record_check(
    "MATCH",
    "match_coverage",
    "PASS" if missing_team_matches == 0 else "FAIL",
    missing_team_matches,
    "Matches missing from the team-match fact",
)


# ---------------------------------------------------------------------------
# Validate team and opponent perspectives
# ---------------------------------------------------------------------------

expected_perspectives = (
    matches_df.select(
        "match_key",
        F.col("home_team_key").alias("team_key"),
        F.col("away_team_key").alias("opponent_team_key"),
        F.lit(True).alias("is_home"),
    )
    .unionByName(
        matches_df.select(
            "match_key",
            F.col("away_team_key").alias("team_key"),
            F.col("home_team_key").alias("opponent_team_key"),
            F.lit(False).alias("is_home"),
        )
    )
)

actual_perspectives = team_matches_df.select(
    "match_key",
    "team_key",
    "opponent_team_key",
    "is_home",
)

perspective_mismatches = (
    expected_perspectives
    .exceptAll(actual_perspectives)
    .count()
    +
    actual_perspectives
    .exceptAll(expected_perspectives)
    .count()
)

record_check(
    "MATCH",
    "team_opponent_perspectives",
    "PASS" if perspective_mismatches == 0 else "FAIL",
    perspective_mismatches,
    "Difference between expected and actual team perspectives",
)


# ---------------------------------------------------------------------------
# Validate team-match scores against match fact
# ---------------------------------------------------------------------------

expected_scores = (
    matches_df.select(
        "match_key",
        F.col("home_team_key").alias("team_key"),
        F.col("home_goals").alias("expected_goals_for"),
        F.col("away_goals").alias("expected_goals_against"),
    )
    .unionByName(
        matches_df.select(
            "match_key",
            F.col("away_team_key").alias("team_key"),
            F.col("away_goals").alias("expected_goals_for"),
            F.col("home_goals").alias("expected_goals_against"),
        )
    )
)

score_differences = (
    team_matches_df
    .select(
        "match_key",
        "team_key",
        "goals_for",
        "goals_against",
    )
    .join(
        expected_scores,
        on=["match_key", "team_key"],
        how="inner",
    )
    .filter(
        ~F.col("goals_for").eqNullSafe(
            F.col("expected_goals_for")
        )
        | ~F.col("goals_against").eqNullSafe(
            F.col("expected_goals_against")
        )
    )
    .count()
)

record_check(
    "MATCH",
    "team_match_score_consistency",
    "PASS" if score_differences == 0 else "FAIL",
    score_differences,
    "Team perspective scores must equal fact_match scores",
)


# ---------------------------------------------------------------------------
# Validate completed and unfinished match rules
# ---------------------------------------------------------------------------

unfinished_results = (
    team_matches_df
    .filter(
        (F.col("match_status") != "FINISHED")
        & (
            F.col("goals_for").isNotNull()
            | F.col("goals_against").isNotNull()
            | F.col("match_result").isNotNull()
            | F.col("points_earned").isNotNull()
        )
    )
    .count()
)

record_check(
    "BUSINESS_RULE",
    "unfinished_matches_without_results",
    "PASS" if unfinished_results == 0 else "FAIL",
    unfinished_results,
)

finished_missing_results = (
    team_matches_df
    .filter(
        (F.col("match_status") == "FINISHED")
        & (
            F.col("goals_for").isNull()
            | F.col("goals_against").isNull()
            | F.col("match_result").isNull()
            | F.col("points_earned").isNull()
        )
    )
    .count()
)

record_check(
    "BUSINESS_RULE",
    "finished_matches_have_results",
    "PASS" if finished_missing_results == 0 else "FAIL",
    finished_missing_results,
)

invalid_points = (
    team_matches_df
    .filter(
        (
            (F.col("match_result") == "W")
            & (F.col("points_earned") != 3)
        )
        | (
            (F.col("match_result") == "D")
            & (F.col("points_earned") != 1)
        )
        | (
            (F.col("match_result") == "L")
            & (F.col("points_earned") != 0)
        )
    )
    .count()
)

record_check(
    "BUSINESS_RULE",
    "team_match_points_consistency",
    "PASS" if invalid_points == 0 else "FAIL",
    invalid_points,
)


# ---------------------------------------------------------------------------
# Validate season-level team standings
# ---------------------------------------------------------------------------

invalid_standings = (
    season_df
    .filter(
        F.col("matches_played")
        != (
            F.col("matches_won")
            + F.col("matches_drawn")
            + F.col("matches_lost")
        )
    )
    .count()
)

record_check(
    "BUSINESS_RULE",
    "season_match_totals",
    "PASS" if invalid_standings == 0 else "FAIL",
    invalid_standings,
)

invalid_goal_difference = (
    season_df
    .filter(
        F.col("goal_difference")
        != (
            F.col("goals_scored")
            - F.col("goals_conceded")
        )
    )
    .count()
)

record_check(
    "BUSINESS_RULE",
    "season_goal_difference",
    "PASS" if invalid_goal_difference == 0 else "FAIL",
    invalid_goal_difference,
)


# ---------------------------------------------------------------------------
# Reconcile match statistics with team-season statistics
# ---------------------------------------------------------------------------

# Only completed matches are used.
#
# Match statistics and FBref season aggregates come from
# different providers and can reflect different definitions
# or official adjustments.
#
# Differences are reported as INFO, not automatic failures.
#
# Incomplete season coverage is also reported rather than
# being interpreted as a data quality defect.

match_aggregates = (
    team_matches_df
    .filter(F.col("match_status") == "FINISHED")
    .groupBy(
        "team_key",
        "competition_key",
        "season_key",
    )
    .agg(
        F.count("*").alias("match_matches_played"),
        F.sum("goals_for").alias("match_goals_scored"),
        F.sum("goals_against").alias("match_goals_conceded"),
        F.sum("points_earned").alias("match_points"),
    )
)

season_aggregates = season_df.select(
    "team_key",
    "competition_key",
    "season_key",
    "matches_played",
    "goals_scored",
    "goals_conceded",
    "points",
)

reconciled_df = (
    season_aggregates
    .join(
        match_aggregates,
        on=[
            "team_key",
            "competition_key",
            "season_key",
        ],
        how="left",
    )
)

RECONCILIATION_METRICS = {
    "matches_played": "match_matches_played",
    "goals_scored": "match_goals_scored",
    "goals_conceded": "match_goals_conceded",
    "points": "match_points",
}

for season_metric, match_metric in RECONCILIATION_METRICS.items():

    differences_df = (
        reconciled_df
        .filter(
            F.col(match_metric).isNotNull()
            & ~F.col(match_metric).eqNullSafe(
                F.col(season_metric)
            )
        )
        .select(
            "team_key",
            "competition_key",
            "season_key",
            F.col(match_metric).alias("match_value"),
            F.col(season_metric).alias("season_value"),
        )
    )

    difference_count = differences_df.count()

    record_check(
        "RECONCILIATION",
        f"match_vs_season_{season_metric}",
        "INFO" if difference_count > 0 else "PASS",
        difference_count,
        "Cross-source differences require contextual review",
    )

    for row in differences_df.collect():
        record_reconciliation(
            season_key=row["season_key"],
            competition_key=row["competition_key"],
            team_key=row["team_key"],
            metric=season_metric,
            match_value=row["match_value"],
            season_value=row["season_value"],
        )

missing_match_coverage = (
    reconciled_df
    .filter(F.col("match_matches_played").isNull())
    .count()
)

record_check(
    "RECONCILIATION",
    "team_seasons_without_match_coverage",
    "INFO" if missing_match_coverage > 0 else "PASS",
    missing_match_coverage,
    "Team-seasons without corresponding finished match records",
)


# ---------------------------------------------------------------------------
# Report Gold table coverage
# ---------------------------------------------------------------------------

for table_name, df in dataframes.items():

    record_check(
        "COVERAGE",
        f"{table_name}_row_count",
        "INFO",
        row_counts[table_name],
    )

for fact_name in [
    "fact_team_season",
    "fact_match",
    "fact_team_match",
]:

    df = dataframes[fact_name]

    coverage = df.agg(
        F.countDistinct("season_key").alias("seasons"),
        F.countDistinct("competition_key").alias("competitions"),
    ).first()

    record_check(
        "COVERAGE",
        f"{fact_name}_season_competition_coverage",
        "INFO",
        (
            f"{coverage['seasons']} seasons, "
            f"{coverage['competitions']} competitions"
        ),
    )


# ---------------------------------------------------------------------------
# Report match-statistics completeness
# ---------------------------------------------------------------------------

STATISTIC_COLUMNS = [
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_corners",
    "away_corners",
    "home_fouls",
    "away_fouls",
    "home_yellow_cards",
    "away_yellow_cards",
    "home_red_cards",
    "away_red_cards",
]

finished_matches_df = matches_df.filter(
    F.col("match_status") == "FINISHED"
)

finished_count = finished_matches_df.count()

for column_name in STATISTIC_COLUMNS:

    null_count = (
        finished_matches_df
        .filter(F.col(column_name).isNull())
        .count()
    )

    null_percentage = (
        100.0 * null_count / finished_count
        if finished_count > 0
        else 0.0
    )

    record_check(
        "COMPLETENESS",
        f"finished_matches_{column_name}",
        "INFO",
        f"{null_percentage:.2f}%",
        f"{null_count} nulls in {finished_count} finished matches",
    )


# ---------------------------------------------------------------------------
# Build validation report
# ---------------------------------------------------------------------------

report_df = spark.createDataFrame(results)

status_summary_df = (
    report_df
    .groupBy("status")
    .agg(F.count("*").alias("checks"))
    .orderBy("status")
)

display(status_summary_df)

display(
    report_df.orderBy(
        "category",
        "check_name",
    )
)


# ---------------------------------------------------------------------------
# Display source reconciliation differences
# ---------------------------------------------------------------------------

if reconciliation_results:

    reconciliation_df = spark.createDataFrame(
        reconciliation_results
    )

    display(
        reconciliation_df.orderBy(
            F.abs(F.col("difference")).desc(),
            "season_key",
            "competition_key",
        )
    )

else:
    print(
        "No differences found between comparable "
        "match and team-season metrics."
    )


# ---------------------------------------------------------------------------
# Final validation summary
# ---------------------------------------------------------------------------

failed_checks = [
    row for row in results
    if row["status"] == "FAIL"
]

info_checks = [
    row for row in results
    if row["status"] == "INFO"
]

passed_checks = [
    row for row in results
    if row["status"] == "PASS"
]

print("=" * 60)
print("GOLD LAYER END-TO-END VALIDATION")
print("=" * 60)
print(f"Gold tables checked: {len(dataframes)}")
print(f"Checks passed: {len(passed_checks)}")
print(f"Informational checks: {len(info_checks)}")
print(f"Checks failed: {len(failed_checks)}")
print("=" * 60)

if failed_checks:

    for check in failed_checks:
        print(
            f"FAILED: {check['check_name']} "
            f"(observed={check['observed']})"
        )

    raise ValueError(
        f"Gold validation failed: "
        f"{len(failed_checks)} checks failed."
    )

print(
    "GOLD VALIDATION PASSED — "
    "no structural or business-rule failures detected."
)


In [0]:

from pyspark.sql import functions as F

GOLD = "football_data_platform.gold"

team_keys = [79, 93, 88, 126, 90]

display(
    spark.table(f"{GOLD}.dim_team")
    .filter(F.col("team_key").isin(team_keys))
    .select("team_key", "team_name")
    .orderBy("team_key")
)


In [0]:

from pyspark.sql import functions as F

GOLD = "football_data_platform.gold"

team_matches = spark.table(
    f"{GOLD}.fact_team_match"
)

teams = spark.table(
    f"{GOLD}.dim_team"
)

display(
    team_matches
    .filter(
        (F.col("season_key") == 2025)
        & (F.col("competition_key") == "ligue_one")
        & F.col("team_key").isin(126, 90)
    )
    .join(
        teams.select(
            F.col("team_key"),
            F.col("team_name"),
        ),
        on="team_key",
        how="left",
    )
    .select(
        "team_name",
        "team_key",
        "match_date",
        "match_key",
        "opponent_team_key",
        "home_away",
        "match_status",
        "goals_for",
        "goals_against",
        "points_earned",
    )
    .orderBy("team_key", "match_date")
)


In [0]:

from pyspark.sql import functions as F

matches = spark.table(
    "football_data_platform.gold.fact_match"
)

display(
    matches
    .filter(
        (F.col("match_status") == "FINISHED")
        & F.col("home_shots").isNull()
    )
    .select(
        "season_key",
        "competition_key",
        "match_date",
        "home_team_key",
        "away_team_key",
        "home_goals",
        "away_goals",
        "home_shots",
        "away_shots",
        "has_csv_source",
        "has_api_source",
    )
)
